In [ ]:
import pandas as pd
df_train = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
df_test = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')
df_raw = pd.concat([df_train,df_test],axis = 0)
df_raw_clean = df_raw[['id','text', 'target']]

df_train_final = df_raw_clean[df_raw_clean['target'].notna()].copy()
df_test_final = df_raw_clean[df_raw_clean['target'].isna()].copy()
df_train_final['target'] = df_train_final['target'].astype(int)

# ============================
# 1. 依存関係のインポート
# ============================
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
from datasets import Dataset
import torch
import os
import re # 正規表現用
import string # 文字列定数用
import html # HTMLエンティティデコード用
import nltk # 自然言語処理ツールキット
from nltk.corpus import wordnet # 同義語置換用 (データ拡張用だが、元のコードに存在)
from nltk.corpus import stopwords # ストップワード用 (データ拡張用だが、元のコードに存在)
import random # ランダム操作用 (データ拡張用だが、元のコードに存在)


# --- 評価指標計算のための追加インポート ---
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np # metrics計算で必要になるため
# --- 追加インポート終わり ---

# NLTKデータのダウンロード (Kaggle環境では通常プリインストール済みだが、念のため)
nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('omw-1.4', quiet=True) # wordnetの依存関係
print("NLTKデータチェック完了 (wordnet, stopwords)。")


# ============================
# 2. データ準備とクリーニング
# ============================
train_df = df_train_final[['text', 'target']].copy()
test_df = df_test_final[['id', 'text']].copy()

# --- テキストクリーニング関数を定義 ---
def clean_text(text):
    text = str(text)
    text = html.unescape(text)
    text = re.sub(r'http\S+', '', text) # URLを除去
    text = re.sub(r'@\w+', '', text)   # メンションを除去
    text = re.sub(r'#', '', text)      # ハッシュタグ記号を除去
    text = text.lower()                # 全て小文字に変換
    text = text.translate(str.maketrans('', '', string.punctuation)) # 句読点を除去
    text = re.sub(r'\s+', ' ', text).strip() # 複数の空白を1つにまとめ、前後の空白を削除
    return text

# クリーニングを適用
train_df['cleaned_text'] = train_df['text'].apply(clean_text)
test_df['cleaned_text'] = test_df['text'].apply(clean_text)

print("データ準備とクリーニング完了。")


# ============================
# 3. 検証セットの分割
# ============================
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df['cleaned_text'].tolist(), # 修正点: クリーニング済みテキストを使用
    train_df['target'].tolist(),
    test_size=0.2,
    random_state=42
)

# ============================
# 4. トークナイザーの読み込みとエンコード
# ============================
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True)

train_dataset = Dataset.from_dict({"text": train_texts, "label": train_labels})
val_dataset = Dataset.from_dict({"text": val_texts, "label": val_labels})
test_dataset = Dataset.from_dict({"text": test_df['cleaned_text'].tolist()}) # 修正点: クリーニング済みテストデータを使用

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

print("トークナイザーの読み込みとエンコード完了。")

# ============================
# 5. DeBERTaモデルの読み込み
# ============================
model = AutoModelForSequenceClassification.from_pretrained("microsoft/deberta-v3-base", num_labels=2)

print("DeBERTaモデルの読み込み完了。")

# ============================
# 6. 訓練パラメータの設定
# ============================
training_args = TrainingArguments(
    output_dir="./tmp",
    do_train=True,
    do_eval=True,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    save_strategy="no",     # モデルは保存しない
    disable_tqdm=False
)

print("訓練パラメータの設定完了。")

# ============================
# 7. Trainerの設定
# ============================

# --- 評価指標計算関数を定義 ---
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    # 適合率、再現率、F1スコア (weighted average)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted', zero_division=0)
    
    # 精度
    acc = accuracy_score(labels, predictions)
    
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }
# --- 評価指標計算関数終わり ---

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics # compute_metrics関数をTrainerに渡す
)

print("Trainerの設定完了。")

# ============================
# 8. 訓練開始
# ============================
trainer.train()

print("訓練完了。")

# --- 検証セットの最終評価を明示的に実行し、出力 ---
print("\n--- 検証セットの最終評価 ---")
eval_results = trainer.evaluate()
print(eval_results)
print("検証セットの最終評価完了。")
# --- 最終評価終わり ---

# ============================
# 9. テストセットの予測
# ============================
predictions = trainer.predict(test_dataset)
pred_labels = torch.argmax(torch.tensor(predictions.predictions), axis=1).numpy()

print("テストセットの予測完了。")

# ============================
# 10. 提出ファイルの保存
# ============================
test_df['target'] = pred_labels
test_df[['id', 'target']].to_csv("submission.csv", index=False)

# ✅ 他の出力をクリーンアップし、提出ファイルのみを保持
for f in os.listdir():
    if f != "submission.csv":
        if os.path.isdir(f):
            os.system(f"rm -rf {f}")
        else:
            os.remove(f)

print("✅ submission.csv が生成され、他のファイルはクリーンアップされました。")
# best
